In [1]:
# Cell 1: Setup & Project Path
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import configure_spark_with_delta_pip

# สร้าง Spark Session ด้วยค่าคอนฟิกที่ถูกต้องตามคู่มือ Delta Lake
builder = SparkSession.builder \
    .appName("Bronze to Silver") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# ใช้ configure_spark_with_delta_pip เพื่อโหลด JARs อัตโนมัติ
spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/02/20 11:40:18 WARN Utils: Your hostname, RATCHANON-PAN.local resolves to a loopback address: 127.0.0.1; using 10.24.10.74 instead (on interface en0)
26/02/20 11:40:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/ratchanon.pan/.ivy2/cache
The jars for the packages stored in: /Users/ratchanon.pan/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-df25f5fa-0c90-481a-88bd-a2938903bed8;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/ratchanon.pan/ecommerce-analytics/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 145ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-df25f5fa-0c90-481a-88bd-a2938903bed8
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/7ms)
26/02/20 11:40:18

In [2]:
# ฟังก์ชันช่วยหา Project Root เพื่อแก้ปัญหา PathNotFound
def get_project_root():
    path = os.path.abspath(os.getcwd())
    while os.path.basename(path) != 'ecommerce-analytics':
        new_path = os.path.dirname(path)
        if new_path == path: break
        path = new_path
    return path

project_root = get_project_root()

In [3]:
# Cell 2: Read Bronze (CSV)
# ใช้ os.path.join เพื่อความแม่นยำของระบบไฟล์
def get_bronze_path(file_name):
    return os.path.join(project_root, "data", "bronze", "olist", file_name)

df_orders = spark.read.csv(get_bronze_path("olist_orders_dataset.csv"), header=True, inferSchema=True)
df_items = spark.read.csv(get_bronze_path("olist_order_items_dataset.csv"), header=True, inferSchema=True)
df_customers = spark.read.csv(get_bronze_path("olist_customers_dataset.csv"), header=True, inferSchema=True)
df_products = spark.read.csv(get_bronze_path("olist_products_dataset.csv"), header=True, inferSchema=True)

In [4]:
# Cell 3: Data Quality Checks
print(f"Total orders: {df_orders.count()}")
df_orders.select([count(when(col(c).isNull(), c)).alias(c) for c in df_orders.columns]).show()

Total orders: 99441
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [5]:
# Cell 4: Clean & Transform - Orders (Silver)
df_orders_silver = df_orders \
    .filter(col("order_status").isNotNull()) \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_delivered_timestamp", to_timestamp("order_delivered_customer_date")) \
    .withColumn("ingestion_date", current_timestamp()) \
    .select("order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_delivered_timestamp", "ingestion_date")

In [6]:
# Cell 5: Write to Silver (Delta format)
# บันทึกข้อมูลในรูปแบบ Delta ตามคู่มือ
silver_orders_path = os.path.join(project_root, "data", "silver", "orders")
df_orders_silver.write.format("delta").mode("overwrite").save(silver_orders_path)
print(f"✅ Orders written to Silver layer at: {silver_orders_path}")

26/02/20 11:41:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ Orders written to Silver layer at: /Users/ratchanon.pan/ecommerce-analytics/data/silver/orders


In [9]:
# Cell 6: Clean - Customers (Silver) 
df_customers_silver = df_customers \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("ingestion_date", current_timestamp()) \
    .dropDuplicates(["customer_id"])

silver_customers_path = os.path.join(project_root, "data", "silver", "customers")
df_customers_silver.write.format("delta").mode("overwrite").save(silver_customers_path)
print(f"✅ Customers written to Silver layer at: {silver_customers_path}")

26/02/20 11:42:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ Customers written to Silver layer at: /Users/ratchanon.pan/ecommerce-analytics/data/silver/customers


In [ ]:
# Cell 7: Verify
df_silver_orders = spark.read.format("delta").load(silver_orders_path)
print(f"Silver orders count: {df_silver_orders.count()}")
df_silver_orders.show(5)

Silver orders count: 99441
+--------------------+--------------------+------------+------------------------+-------------------------+--------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_delivered_timestamp|      ingestion_date|
+--------------------+--------------------+------------+------------------------+-------------------------+--------------------+
|d9eabc69f974b3d08...|d4be0795e8fa7ea79...|   delivered|     2018-03-21 19:32:50|      2018-03-28 18:56:52|2026-02-20 11:41:...|
|c9fbf88f9c58364e0...|7ec5b53960d508118...|   delivered|     2018-02-05 16:47:14|      2018-02-20 19:59:00|2026-02-20 11:41:...|
|fd01a48a7d75383a3...|014f2d069b53eec84...|   delivered|     2017-08-18 14:17:44|      2017-08-28 17:04:11|2026-02-20 11:41:...|
|6d4616de4341417e1...|adc1b0d30fe2b52bd...|   delivered|     2017-04-15 11:46:19|      2017-05-05 15:56:20|2026-02-20 11:41:...|
|c606769bddf9fb8b9...|a4b29e455132615d4...|   delivered|     2017-06-1

26/02/20 12:46:30 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 925173 ms exceeds timeout 120000 ms
26/02/20 12:46:30 WARN SparkContext: Killing executors is not supported by current scheduler.
26/02/20 12:46:34 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$